### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

## 1. Understanding the Dataset

In [ ]:
df_raw = pd.read_csv("/content/titanic.csv")
print("Dataset Shape:", df_raw.shape)
print("\nDataset Columns and Types:")
print(df_raw.dtypes)
print("\nMissing values count:")
print(df_raw.isnull().sum())

df = df_raw.copy()

## 2. Checking and Handling Missing Values

In [ ]:
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)

df['Has_Cabin'] = df['Cabin'].notnull().astype(int)

print("Missing values after handling:")
print(df[['Age', 'Embarked', 'Has_Cabin']].isnull().sum())

## 3. Removing Duplicate Records

In [ ]:
duplicates = df.duplicated().sum()
print("Duplicate records found:", duplicates)
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")

## 4. Detecting and Handling Outliers

In [ ]:
for col in ['Age', 'Fare']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
    print(f"{col}: Found {outliers_count} outliers. Clipping to range [{lower_bound:.2f}, {upper_bound:.2f}]")
    
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

## 5. Handling Incorrect Data Types

In [ ]:
df['Pclass'] = df['Pclass'].astype(str)
df['Sex'] = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')

print("Updated Data Types:")
print(df[['Pclass', 'Sex', 'Embarked']].dtypes)

## 6. Handling Categorical Variables

In [ ]:
# Label Encoding for Binary Variable (Sex)
df['Sex_Encoded'] = df['Sex'].map({'male': 1, 'female': 0}).astype(int)

# Ordinal Encoding for Pclass
df['Pclass_Encoded'] = df['Pclass'].map({'1': 3, '2': 2, '3': 1}).astype(int)

# One-Hot Encoding for Nominal Variable (Embarked)
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True, dtype=int)
df = pd.concat([df, embarked_dummies], axis=1)

print("Categorical encoding completed.")

## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[['Age', 'Fare']])
df['Age_Scaled'] = scaled_features[:, 0]
df['Fare_Scaled'] = scaled_features[:, 1]

print("StandardScaler applied to Age and Fare.")

## 8. Removing Irrelevant or Redundant Features

In [ ]:
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Pclass', 'Sex', 'Embarked', 'Age', 'Fare']
cleaned_df = df.drop(columns=cols_to_drop)

print("Retained columns:", list(cleaned_df.columns))

## 9. Handling Skewness

In [ ]:
print("Fare skewness before transform:", df['Fare'].skew())

cleaned_df['Fare_Log_Scaled'] = np.log1p(df['Fare'])
cleaned_df['Fare_Log_Scaled'] = scaler.fit_transform(cleaned_df[['Fare_Log_Scaled']])
cleaned_df = cleaned_df.drop(columns=['Fare_Scaled'])

print("Fare skewness after log-transform:", np.log1p(df['Fare']).skew())
print("Preprocessed Shape:", cleaned_df.shape)

## Save Preprocessed Dataset

In [ ]:
cleaned_df.to_csv('titanic_preprocessed.csv', index=False)
print("Preprocessed dataset saved to 'titanic_preprocessed.csv'.")
cleaned_df.head()